## HR Decision Support Assistant: AI-Powered Final Candidates Evaluation and Ranking

In [47]:
import os
import json
import random

from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from groq import Groq

### General configuration (AIP, Model,...)

In [48]:
load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("Please set the GROQ_API_KEY variable in the environment or .env file")

client = Groq(api_key=api_key)

MODEL_NAME = "openai/gpt-oss-120b"
TEMPERATURE = 0.0

N_STABILITY_RUNS = 3
RUN_BIAS_AUDIT = True

INPUT_FILE = "candidates_template.xlsx"
OUTPUT_XLSX = "hiring_recommendation.xlsx"
AUDIT_LOG_FILE = "audit_log.jsonl"

### project setting

In [49]:
JOB_DESCRIPTION = """
Job Title: Senior Data Analyst

Key Responsibilities:
- Design, build, and maintain automated dashboards in Power BI and Tableau.
- Write complex SQL queries for data extraction and data cleaning.
- Analyze candidate evaluation data using Python (Pandas, Scikit-learn).

Qualifications & Requirements:
- 4+ years of experience in data analytics or business intelligence.
- Strong proficiency in SQL, Python, and Excel.
- Experience with A/B testing and algorithmic bias evaluation is a plus.
"""

SCORING_CRITERIA = {
    "skills_experience_fit":   {"label": "Skills & Experience Alignment",    "weight": 0.35},
    "interview_performance":   {"label": "Interview Assessment",            "weight": 0.30},
    "salary_fit":              {"label": "Salary Expectations & Budget Fit", "weight": 0.20},
    "manager_notes_alignment": {"label": "Manager Notes & Role Alignment",   "weight": 0.15},
}

total_weight = sum(cfg["weight"] for cfg in SCORING_CRITERIA.values())
assert abs(total_weight - 1) < 1e-6, f"sum of {total_weight} is 1"

### prompt setting

In [50]:
def build_system_prompt():
    criteria_lines = "\n".join(
        f'- {key}: {cfg["label"]} (0 to 100)'
        for key, cfg in SCORING_CRITERIA.items()
    )
    json_keys = ", ".join(f'"{k}": 0' for k in SCORING_CRITERIA)

    return f"""
You are a Senior HR Advisor assisting a hiring team in comparing final candidates for the offer.

TARGET JOB DESCRIPTION:
{JOB_DESCRIPTION}

INSTRUCTIONS:
1. Evaluate candidates strictly against the Job Description.
2. Score each candidate independently (0 to 100) on these criteria:
{criteria_lines}
3. Ignore gender, age, ethnicity, religion, or personal background.
4. If data is missing for a criterion, use score 50 and flag it in "insufficient_data_flags".
5. Return strictly valid JSON with this format:

{{
  "candidates": [
    {{
      "name": "Exact candidate name",
      "criteria_scores": {{{json_keys}}},
      "strengths": ["..."],
      "concerns": ["..."],
      "notes": "...",
      "insufficient_data_flags": ["..."]
    }}
  ],
  "summary": "Comparative summary",
  "key_tradeoffs": "Key trade-offs among candidates"
}}
""".strip()


def build_candidate_block(row):
    return f"""
Candidate Name: {row.get('Candidate_Name', '-')}
Interview Score: {row.get('Interview_Score', '-')}
Years of Experience: {row.get('Years_Experience', '-')}
Manager Notes: {row.get('Manager_Notes', '-')}
Salary Range: {row.get('Company_Salary_Range', '-')}
Salary Expectation: {row.get('Candidate_Salary_Expectation', '-')}
Resume Summary: {row.get('Resume_Summary', '-')}
Other Notes: {row.get('Other_Notes', '-')}
""".strip()

### Model call + log

In [51]:
def log_audit(run_type, df, raw_response, success):
    entry = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "run_type": run_type,
        "model": MODEL_NAME,
        "candidate_order": df["Candidate_Name"].tolist(),
        "parsed_ok": success,
        "raw_response": raw_response,
    }
    with open(AUDIT_LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")


def call_llm(df, run_type="main"):
    candidates_text = "\n\n---\n\n".join(build_candidate_block(r) for _, r in df.iterrows())
    user_prompt = f"Candidates to evaluate:\n\n{candidates_text}"

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": build_system_prompt()},
                {"role": "user", "content": user_prompt},
            ],
            temperature=TEMPERATURE,
            response_format={"type": "json_object"},
        )
        raw_text = response.choices[0].message.content
        data = json.loads(raw_text)
        log_audit(run_type, df, raw_text, success=True)
        return data
    except Exception as e:
        log_audit(run_type, df, str(e), success=False)
        print(f"Warning: Call '{run_type}' failed: {e}")
        return {"candidates": [], "summary": "", "key_tradeoffs": "", "error": str(e)}

### Waighted score calculation and ranking

In [52]:
def compute_rankings(result):
    rankings = []
    for c in result.get("candidates", []):
        scores = c.get("criteria_scores", {})
        total = sum(scores.get(k, 50) * cfg["weight"] for k, cfg in SCORING_CRITERIA.items())
        rankings.append((c["name"], round(total, 2)))
    return sorted(rankings, key=lambda x: x[1], reverse=True)

### Staibility test

In [53]:
def run_stability_test(df, n_runs):
    history = {name: [] for name in df["Candidate_Name"]}

    for i in range(n_runs):
        shuffled = df.sample(frac=1).reset_index(drop=True)
        result = call_llm(shuffled, run_type=f"stability_run_{i + 1}")
        ranking = compute_rankings(result)
        for rank, (name, _) in enumerate(ranking, start=1):
            if name in history:
                history[name].append(rank)

    summary = {}
    for name, ranks in history.items():
        if ranks:
            spread = max(ranks) - min(ranks)
            summary[name] = {
                "ranks": ranks,
                "spread": spread,
                "unstable": spread >= 2,
            }
    return summary

### Bias audit

In [54]:
def run_bias_audit(df, real_result):
    df_anon = df.copy().reset_index(drop=True)
    mapping = {}
    for i in range(len(df_anon)):
        anon_label = f"Candidate {i + 1}"
        mapping[anon_label] = df_anon.loc[i, "Candidate_Name"]
        df_anon.loc[i, "Candidate_Name"] = anon_label

    anon_result = call_llm(df_anon, run_type="bias_audit_anonymized")

    real_order = {name: rank for rank, (name, _) in enumerate(compute_rankings(real_result), 1)}
    anon_order = {}
    if anon_result.get("candidates"):
        anon_order = {
            mapping.get(name, name): rank
            for rank, (name, _) in enumerate(compute_rankings(anon_result), 1)
        }

    audit = {}
    for name in real_order:
        r_rank = real_order.get(name)
        a_rank = anon_order.get(name)
        audit[name] = {
            "real_rank": r_rank,
            "anon_rank": a_rank,
            "impacted": r_rank is not None and a_rank is not None and r_rank != a_rank,
        }
    return audit

### Creating the excel file

In [55]:
def export_excel_report(main_result, stability, bias_audit, output_path):
    candidates = main_result.get("candidates", [])

    # sheet1
    ranking_data = []
    for c in candidates:
        row = {"Candidate Name": c["name"]}
        scores = c.get("criteria_scores", {})
        total_score = 0
        for key, cfg in SCORING_CRITERIA.items():
            score = scores.get(key, 50)
            column_name = f'{cfg["label"]} ({cfg["weight"]:.0%})'
            row[column_name] = score
            total_score += score * cfg["weight"]
        row["Final Weighted Score"] = round(total_score, 2)
        ranking_data.append(row)

    df_ranking = pd.DataFrame(ranking_data)
    if not df_ranking.empty:
        df_ranking["Rank"] = df_ranking["Final Weighted Score"].rank(ascending=False, method="min").astype(int)
        df_ranking = df_ranking.sort_values("Rank")

    # sheet 2
    details_data = [
        {
            "Candidate Name": c["name"],
            "Strengths": "\n".join(c.get("strengths", [])),
            "Concerns": "\n".join(c.get("concerns", [])),
            "Notes": c.get("notes", ""),
            "Missing Data Flags": ", ".join(c.get("insufficient_data_flags", [])),
        }
        for c in candidates
    ]
    df_details = pd.DataFrame(details_data)

    # sheet 3
    audit_data = [
        {
            "Candidate Name": c["name"],
            "Ranks Across Runs": str(stability.get(c["name"], {}).get("ranks", "—")),
            "Rank Spread": stability.get(c["name"], {}).get("spread", "—"),
            "Unstable?": "Yes" if stability.get(c["name"], {}).get("unstable") else "No",
            "Rank (Real Name)": bias_audit.get(c["name"], {}).get("real_rank", "—"),
            "Rank (Anonymized)": bias_audit.get(c["name"], {}).get("anon_rank", "—"),
            "Bias Detected?": "Yes" if bias_audit.get(c["name"], {}).get("impacted") else "No",
        }
        for c in candidates
    ]
    df_audit = pd.DataFrame(audit_data)

    # sheet 4
    summary_data = [
        {"Section": "Job Description", "Details": JOB_DESCRIPTION.strip()},
        {"Section": "Executive Summary", "Details": main_result.get("summary", "")},
        {"Section": "Key Trade-offs", "Details": main_result.get("key_tradeoffs", "")},
    ]
    df_summary = pd.DataFrame(summary_data)

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        df_ranking.to_excel(writer, sheet_name="Ranking", index=False)
        df_details.to_excel(writer, sheet_name="Details", index=False)
        df_audit.to_excel(writer, sheet_name="Stability & Bias Audit", index=False)
        df_summary.to_excel(writer, sheet_name="Executive Summary", index=False)

### Execution

In [56]:
def main():
    if not Path(INPUT_FILE).exists():
        print(f"Error: '{INPUT_FILE}' not found.")
        return

    df = pd.read_excel(INPUT_FILE)
    df.columns = df.columns.astype(str).str.strip()

    print(f"Analyzing {len(df)} candidate(s)...")
    main_result = call_llm(df, run_type="main")
    if main_result.get("error"):
        print("Evaluation failed.")
        return

    print("Running stability audit...")
    stability = run_stability_test(df, N_STABILITY_RUNS)

    bias_audit = {}
    if RUN_BIAS_AUDIT:
        print("Running name bias audit...")
        bias_audit = run_bias_audit(df, main_result)

    export_excel_report(main_result, stability, bias_audit, OUTPUT_XLSX)
    print(f"Process complete. Report exported to: {OUTPUT_XLSX}")


main()

Analyzing 3 candidate(s)...
Running stability audit...
Running name bias audit...
Process complete. Report exported to: hiring_recommendation.xlsx
